In [101]:
import pandas as pd
from collections import Counter
from rapidfuzz import fuzz
import numpy as np
from tqdm import tqdm
import re
from utils.utils_oab import score_fuzz

In [102]:
oab_clt = pd.read_csv('csvs/oab/oab_questions_unified_no_answer.csv')

In [103]:
oab_all = pd.read_csv('csvs/oab/oab_questions_all.csv')

In [105]:
oab_all = oab_all.drop_duplicates()
oab_all = oab_all.reset_index(drop=True)

In [106]:
match_dict = {}
dict_count = []
for i,r in oab_clt.iterrows():
    mask = oab_all[oab_all['question'] == r['question']]
    if len(mask) == 0:
        match_dict[r['question_number']] = None
        dict_count.append(0)
    if len(mask) == 1:
        match_dict[r['question_number']] = mask.iloc[0]
        dict_count.append(1)
    if len(mask) > 1:
        match_dict[r['question_number']] = 'Dupla ocorrência'
        dict_count.append(2)

In [107]:
for i, r in tqdm(oab_clt.iterrows()):
    if match_dict[i+1] is None:
        score = score_fuzz(r['question'], oab_all)

        if len(score) == 1:
            match_dict[i+1] = oab_all.iloc[score[0]]
            dict_count[i] = 1

        elif len(score) > 1:
            match_dict[i+1] = 'Dupla ocorrência'
            dict_count[i] = 2

        else:
            match_dict[i+1] = None

288it [00:03, 81.88it/s] 


In [108]:
match_dict

{1: question_number                                                   76
 question           Jorge Lucas trabalhou por um ano na sociedade ...
 options            ['Tendo sido pago o valor do aviso prévio espo...
 exam               FGV - 2026 - OAB - Exame da Ordem Unificado XL...
 answer                                                             C
 Name: 2795, dtype: object,
 2: question_number                                                   75
 question           Jonas trabalhava em uma sociedade empresária d...
 options            ['Nada há a fazer, porque é direito do emprega...
 exam               FGV - 2026 - OAB - Exame da Ordem Unificado XL...
 answer                                                             C
 Name: 2794, dtype: object,
 3: question_number                                                   74
 question           A sociedade empresária Distribuidora de Bebida...
 options            ['Todos os benefícios concedidos não integrarã...
 exam               FGV -

In [109]:
dict_count = []
for i in match_dict.values():
    if i is None:
        dict_count.append(0)
    elif type(i) == pd.Series:
        dict_count.append(1)
    else:
        dict_count.append(2)

In [110]:
Counter(dict_count)

Counter({1: 288})

In [ ]:
for i in match_dict.keys():
    if match_dict[i] is None:
        print(f"Question {i} not found")

In [113]:
oab_with_answer = []
for i in match_dict.keys():
    if type(match_dict[i]) == pd.Series:
        oab_with_answer.append({
            'question_number': int(match_dict[i]['question_number']),
            'question': match_dict[i]['question'],
            'options': match_dict[i]['options'],
            'answer': match_dict[i]['answer'],
            'exam': match_dict[i]['exam']
        })

In [ ]:
df_oab_with_answer = pd.DataFrame(oab_with_answer)
# Annulled questions (without answer in the answer key) are not saved
annulled = df_oab_with_answer[df_oab_with_answer['answer'].isna()]
df_oab_with_answer = df_oab_with_answer[df_oab_with_answer['answer'].notna()].reset_index(drop=True)
print(f'Annulled questions removed: {len(annulled)}')
for exam, numbers in annulled.groupby('exam')['question_number']:
    print(f'  {exam}: {numbers.tolist()}')

In [118]:
df_oab_with_answer.to_csv('csvs/oab/oab_questions_with_answer.csv', index=False)